# 🛠️ Tool-Calling Agents with LangGraph

## Learning Objectives
In this notebook, you will learn:
1. **Defining Tools** - How to wrap plain Python functions as LangGraph-compatible tools with the `@tool` decorator
2. **Agent State** - How to model conversation history with `TypedDict` and the `add_messages` reducer
3. **The Tool-Calling Loop** - How to wire an `agent` node and a `ToolNode` together with conditional edges so the LLM can call tools and see their results
4. **Execution Tracing** - How to inspect the full message trace of an agent run to understand what happened at each step
5. **Error Handling** - How tool-level errors (like division by zero) surface back to the LLM instead of crashing the graph

## Prerequisites
- Completion of the earlier LangGraph Fundamentals notebooks (state, graphs, routing, tools)
- Familiarity with LangChain's `@tool` decorator and `bind_tools()`
- An `OPENAI_API_KEY` set in a `.env` file at the project root

---
## 📦 Part 1: Environment Setup

We start by importing the LangGraph and LangChain building blocks we need, loading environment variables from `.env`, and initializing the LLM that will power the agent.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and LLM Initialization
# ============================================================================
from typing import Literal

from dotenv import load_dotenv
from typing_extensions import Annotated, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

# Load API keys from the project's .env file
load_dotenv()

# Initialize the LLM that will decide which tools to call
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

print(f"🤖 LLM initialized: {llm.model_name}")

---
## 🧰 Part 2: Defining Tools

Before an agent can act, it needs tools to act with. Each tool below is a plain Python function decorated with `@tool`, which turns its signature and docstring into the schema the LLM uses to decide when and how to call it.

### Key Concepts:
- **`@tool` decorator**: Wraps a function so LangChain can expose its name, docstring, and type hints to the LLM as a callable schema
- **Simulated data**: These tools return canned/simulated results (no real API calls) so the notebook runs deterministically without extra API keys

### 2.1 🧮 `calculate` — Arithmetic Tool

A simple tool that evaluates a mathematical expression string. Note the `eval()` call is for demonstration only — production code should use a safe math parser instead.

In [ ]:
# ============================================================================
# TOOL: calculate - Evaluate a Mathematical Expression
# ============================================================================
@tool
def calculate(expression: str) -> str:
    """Calculate a mathematical expression. Example: calculate('2 + 2')"""
    try:
        result = eval(expression)  # Note: In production, use a safe math parser
        return f"The result of {expression} is {result}"
    except Exception as e:
        return f"Error calculating: {e}"

### 2.2 🌤️ `get_weather` — Weather Lookup Tool

A tool that looks up weather for a small set of hardcoded cities. Cities outside the lookup table return a friendly "not available" message instead of raising an error.

In [ ]:
# ============================================================================
# TOOL: get_weather - Simulated Weather Lookup
# ============================================================================
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    # Simulated weather data
    weather_data = {
        "new york": "72°F, Sunny",
        "london": "58°F, Cloudy",
        "tokyo": "68°F, Clear",
        "paris": "65°F, Partly Cloudy",
    }
    city_lower = city.lower()
    if city_lower in weather_data:
        return f"Weather in {city}: {weather_data[city_lower]}"
    return f"Weather data not available for {city}"

### 2.3 🔍 `search_web` — Simulated Web Search Tool

A tool that simulates a web search by matching the query against a small hardcoded results table.

In [ ]:
# ============================================================================
# TOOL: search_web - Simulated Web Search
# ============================================================================
@tool
def search_web(query: str) -> str:
    """Simulate a web search for a query."""
    # Simulated search results
    search_results = {
        "python programming": "Python is a high-level programming language known for its readability and versatility.",
        "latest news": "Today's top news: AI continues to advance, impacting various industries worldwide.",
        "best restaurants in new york": "Top restaurants in New York include Le Bernardin, Per Se, and Eleven Madison Park.",
    }
    query_lower = query.lower()
    if query_lower in search_results:
        return f"Search results for '{query}': {search_results[query_lower]}"
    return f"No search results found for '{query}'"

---
## 🧩 Part 3: Agent State and Graph Construction

With the tools defined, we now model the agent's conversational state and wire together the graph that lets the LLM call those tools in a loop.

### Key Concepts:
- **`AgentState`**: A `TypedDict` holding the running list of messages
- **`add_messages` reducer**: Appends new messages to state instead of overwriting it, which is what lets the conversation accumulate across agent ↔ tool round trips
- **Conditional routing**: The `should_continue` function inspects the last message to decide whether to route to the `tools` node or end the graph

### 3.1 `AgentState` — Conversation State Schema

In [ ]:
# ============================================================================
# STATE: AgentState - Message History Schema
# ============================================================================
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

### 3.2 🏗️ `create_tool_agent` — Building the Tool-Calling Loop

This function assembles the full agent graph:
1. Bind the tools to the LLM with `bind_tools()` so it can emit tool calls
2. An `agent` node that invokes the LLM
3. A `tools` node (`ToolNode`) that executes whichever tools the LLM requested
4. Conditional edges that loop back to the `agent` node after tool execution, until the LLM responds without any tool calls

In [ ]:
# ============================================================================
# GRAPH: create_tool_agent - Assemble the Tool-Calling Agent Graph
# ============================================================================
def create_tool_agent():
    """Create a basic tool-calling agent."""
    tools = [calculate, get_weather, search_web]
    llm_with_tools = llm.bind_tools(tools)  # this is the secret!

    def agent_node(state: AgentState) -> dict:
        # Generate a response using the LLM with tool access
        response = llm_with_tools.invoke(state["messages"])
        return {"messages": [response]}

    def should_continue(state: AgentState) -> Literal["tools", "end"]:
        """Check if we should continue to tools or end."""
        last_message = state["messages"][-1]
        # If no tool calls, we're done
        if not hasattr(last_message, "tool_calls") or not last_message.tool_calls:
            return "end"
        return "tools"

    # create tool node
    tool_node = ToolNode(tools)

    # create graph
    graph = StateGraph(AgentState)

    # add nodes and edges
    graph.add_node("agent", agent_node)
    graph.add_node("tools", tool_node)
    graph.add_edge(START, "agent")
    graph.add_conditional_edges(
        "agent", should_continue, {"tools": "tools", "end": END}
    )
    graph.add_edge("tools", "agent")  # loop back after tool execution

    return graph.compile()


print("✅ create_tool_agent() defined")

---
## ▶️ Part 4: Running the Agent

Now let's exercise the compiled graph with real queries and see how it decides which tools to call, and inspect the full message trace of a run.

### 4.1 💬 `demo_tool_agent` — Basic Query Demo

Runs a handful of queries through the agent and prints the final response plus the total number of messages accumulated in state (user message + any tool calls/results + final answer).

In [ ]:
# ============================================================================
# DEMO: demo_tool_agent - Run a Few Sample Queries
# ============================================================================
def demo_tool_agent():
    """Demo the tool-calling agent."""
    agent = create_tool_agent()

    queries = [
        "What's 25 * 17?",
        "What's the weather in Tokyo?",
        "What's 100 / 4 and what's the weather in London?",
    ]

    print("Tool-Calling Agent Demo:\n")
    for query in queries:
        print(f"Query: {query}")
        result = agent.invoke({"messages": [HumanMessage(content=query)]})

        # Get final response
        final_message = result["messages"][-1]
        print(f"Response: {final_message.content}")
        print(f"Total messages: {len(result['messages'])}")
        print("-" * 40)

### 4.2 🔎 `demo_tool_execution_trace` — Inspecting the Full Message Trace

Rather than just printing the final answer, this demo walks every message in the resulting state — the human query, the AI's tool-call requests, and each tool's returned result — so you can see exactly how the agent loop unfolded.

In [ ]:
# ============================================================================
# DEMO: demo_tool_execution_trace - Walk the Full Message Trace
# ============================================================================
def demo_tool_execution_trace():
    """Show detailed tool execution trace."""
    agent = create_tool_agent()

    print("\nTool Execution Trace:\n")
    result = agent.invoke(
        {
            "messages": [
                HumanMessage(content="Calculate 15% of 250 and check weather in Paris")
            ]
        }
    )

    for i, msg in enumerate(result["messages"]):
        msg_type = type(msg).__name__
        print(f"\n[{i}] {msg_type}:")
        if isinstance(msg, HumanMessage):
            print(f"  Content: {msg.content}")
        elif isinstance(msg, AIMessage):
            if msg.tool_calls:
                print(f"  Tool calls: {len(msg.tool_calls)}")
                for tc in msg.tool_calls:
                    print(f"    - {tc['name']}({tc['args']})")
            else:
                print(f"  Content: {msg.content}")
        elif isinstance(msg, ToolMessage):
            print(f"  Tool: {msg.name}")
            print(f"  Result: {msg.content}")

---
## ⚠️ Part 5: Error Handling in Tools

Tools can fail — divide-by-zero, bad input, a downstream API timeout. A well-written tool catches the failure and returns a descriptive string instead of raising, so the error becomes part of the conversation and the LLM can react to it (e.g. explain the problem to the user) rather than crashing the graph.

### 5.1 ➕ `divide` — A Tool That Can Fail Gracefully

In [ ]:
# ============================================================================
# TOOL: divide - Division With Guarded Zero-Division Handling
# ============================================================================
@tool
def divide(a: float, b: float) -> str:
    """Divide two numbers."""
    if b == 0:
        return "Error: Division by zero"
    result = a / b
    return f"The result of {a} divided by {b} is {result}"

### 5.2 🧪 `demo_tool_with_errors` — Exercising the Error Path

Builds a small standalone agent around just the `divide` tool and runs it against both a valid division and a division by zero, showing that the guarded error message flows back through the graph instead of raising an exception.

In [ ]:
# ============================================================================
# DEMO: demo_tool_with_errors - Build a Minimal Agent and Trigger an Error
# ============================================================================
def demo_tool_with_errors():
    """Demo tool error handling."""
    tools = [divide]
    llm_with_tools = llm.bind_tools(tools)

    def agent_node(state: AgentState) -> dict:
        response = llm_with_tools.invoke(state["messages"])
        return {"messages": [response]}

    def should_continue(state: AgentState) -> Literal["tools", "end"]:
        last_message = state["messages"][-1]
        if not hasattr(last_message, "tool_calls") or not last_message.tool_calls:
            return "end"
        return "tools"

    tool_node = ToolNode(tools)
    graph = StateGraph(AgentState)
    graph.add_node("agent", agent_node)
    graph.add_node("tools", tool_node)
    graph.add_edge(START, "agent")
    graph.add_conditional_edges(
        "agent", should_continue, {"tools": "tools", "end": END}
    )
    graph.add_edge("tools", "agent")
    agent = graph.compile()

    print("\nTool Error Handling Demo:\n")
    queries = [
        "Divide 100 by 5",
        "Divide 100 by 0",  # Will trigger error
    ]
    for query in queries:
        result = agent.invoke({"messages": [HumanMessage(content=query)]})
        print(f"Query: {query}")
        print(f"Response: {result['messages'][-1].content}")
        print("-" * 40)

---
## 🚀 Part 6: Putting It All Together

Jupyter sets `__name__` to `"__main__"`, so the original script's `if __name__ == "__main__":` guard runs as-is here. Uncomment any of the other lines to try the other two demos — they are left commented out to keep a single run's output focused on the error-handling demo.

In [ ]:
# ============================================================================
# RUN: Execute a Demo
# ============================================================================
if __name__ == "__main__":
    # demo_tool_agent()
    # demo_tool_execution_trace()
    demo_tool_with_errors()

---
## 📝 Summary

In this notebook, we learned:

### 1. Defining Tools
- The `@tool` decorator turns a plain Python function's signature and docstring into a schema the LLM can call
- `calculate`, `get_weather`, and `search_web` each demonstrate a self-contained tool with simulated data

### 2. Agent State and Graph Wiring
- `AgentState` uses the `add_messages` reducer so new messages accumulate rather than overwrite
- `create_tool_agent()` wires an `agent` node, a `ToolNode`, and conditional edges into a loop: agent → tools → agent → ... → end

### 3. Running and Tracing the Agent
- `demo_tool_agent()` shows the final response and message count for a batch of queries
- `demo_tool_execution_trace()` walks every message — human, AI tool-call requests, and tool results — to make the agent loop visible

### 4. Error Handling
- Tools should catch failures (like division by zero) and return a descriptive string rather than raising, so the LLM can react to the error within the conversation

### Next Steps
- This is the final notebook in **03_LangGraph_Fundamentals** — you now have the core mechanics: state, graphs, routing, tools, and tool-calling agents
- Continue to **`04_Multi_Agent_Systems/`** to see how multiple specialized agents coordinate with each other on top of these same LangGraph primitives